In [1]:
%pip install dotenv
%pip install pypdf2
%pip install pdfrw
%pip install reportlab
%pip install --upgrade pypdf2

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient
import pandas as pd
from io import StringIO

# Load environment variables from environment.env
load_dotenv("environment.env")

# Get values from env
connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
template_container_name = os.getenv("TEMPLATE_CONTAINER_NAME")

# Initialize client
blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_client = blob_service_client.get_container_client(template_container_name)

# Example: List blobs
print("Blobs in container:")
for blob in container_client.list_blobs():
    print("  -", blob.name)


Blobs in container:
  - ADA Notice to Borrowers-FORM.pdf
  - Acknowledgement of Receipt - FORM.pdf
  - Agreement to Cooperate-FORM.pdf
  - Condominium Rider-FORM.pdf
  - Disclosure Statement-FORM.pdf
  - Escrow Waiver Affidavit-FORM.pdf
  - Info Disclosure Author-FORM.pdf
  - Mortgage-FORM.pdf
  - Perjury Statement-FORM.pdf
  - Primary Residence Affidavit-FORM.pdf
  - Promissory Note-FORM.pdf
  - SSICollect-FORM.pdf


In [27]:
import os
from io import BytesIO, StringIO
import pandas as pd
from pdfrw import PdfReader, PdfWriter, PageMerge
from reportlab.pdfgen import canvas
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv

# --- Load environment variables ---
load_dotenv("environment.env")
connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
sample_container_name = os.getenv("SAMPLE_CONTAINER_NAME")  # CSV container
template_container_name = "fillable-pdfs"                  # template PDFs
output_container_name = "promissory-note-filled-v3"        # output PDFs

# --- Initialize Blob Service Client ---
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

# --- Load CSV from blob into DataFrame ---
csv_blob_name = "Promissory_Note_Dummy_Data__50_rows_.csv"
blob_client = blob_service_client.get_blob_client(container=sample_container_name, blob=csv_blob_name)
data = blob_client.download_blob().readall()
df = pd.read_csv(StringIO(data.decode("utf-8")))

# --- Helper: Get all form fields and positions from template ---
def get_form_fields(template_pdf):
    """
    Returns a dict: {field_name: (page_index, (x0, y0, x1, y1))}
    """
    fields = {}
    for i, page in enumerate(template_pdf.pages):
        annotations = page.Annots
        if annotations:
            for annot in annotations:
                if annot.Subtype and annot.Subtype == "/Widget":
                    field_name = str(annot.T)[1:-1]  # Remove parentheses from /T
                    rect = [float(x) for x in annot.Rect]
                    fields[field_name] = (i, rect)
    return fields

# --- Helper: Create overlay PDFs per page ---
def create_overlay(fields_to_fill, template_pdf):
    """
    Returns a list of overlay PDFs (BytesIO), one per template page.
    Each overlay PDF has exactly one page, even if empty.
    """
    form_fields = get_form_fields(template_pdf)
    overlays = []

    for page_index, page in enumerate(template_pdf.pages):
        packet = BytesIO()
        c = canvas.Canvas(packet, pagesize=(float(page.MediaBox[2]), float(page.MediaBox[3])))

        # Draw only fields on this page
        for field_name, value in fields_to_fill.items():
            if field_name in form_fields:
                field_page_index, rect = form_fields[field_name]
                if field_page_index == page_index:
                    x0, y0, x1, y1 = rect
                    c.drawString(x0 + 2, y0 + 2, str(value))

        c.showPage()  # finalize page
        c.save()
        packet.seek(0)
        overlays.append(packet)

    return overlays

# --- Process each row in the CSV ---
for _, row in df.iterrows():
    loan_number = str(row["Loan Number"])
    print(f"\n📝 Filling Promissory Note for Loan {loan_number}")

    # Download template PDF
    template_blob_client = blob_service_client.get_blob_client(
        container=template_container_name,
        blob="Promissory Note-FORM.pdf"
    )
    template_pdf_data = template_blob_client.download_blob().readall()
    template_pdf = PdfReader(BytesIO(template_pdf_data))

    # Prepare fields
    fields_to_fill = {col: str(row[col]) if pd.notna(row[col]) else "" for col in df.columns}

    # Create overlay PDFs per page
    overlay_streams = create_overlay(fields_to_fill, template_pdf)
    overlay_pdfs = [PdfReader(s) for s in overlay_streams]

    # Merge each overlay with the corresponding page
    for page_index, page in enumerate(template_pdf.pages):
        overlay_page = overlay_pdfs[page_index].pages[0]
        merger = PageMerge(page)
        merger.add(overlay_page).render()

    # Save flattened PDF to memory
    output_stream = BytesIO()
    PdfWriter().write(output_stream, template_pdf)
    output_stream.seek(0)

    # Upload to Azure Blob Storage
    output_blob_name = f"{loan_number}.pdf"
    output_blob_client = blob_service_client.get_blob_client(
        container=output_container_name,
        blob=output_blob_name
    )
    output_blob_client.upload_blob(output_stream, overwrite=True)
    print(f"✅ Uploaded flattened PDF as {output_container_name}/{output_blob_name}")



📝 Filling Promissory Note for Loan LN-100000
✅ Uploaded flattened PDF as promissory-note-filled-v3/LN-100000.pdf

📝 Filling Promissory Note for Loan LN-100001
✅ Uploaded flattened PDF as promissory-note-filled-v3/LN-100001.pdf

📝 Filling Promissory Note for Loan LN-100002
✅ Uploaded flattened PDF as promissory-note-filled-v3/LN-100002.pdf

📝 Filling Promissory Note for Loan LN-100003
✅ Uploaded flattened PDF as promissory-note-filled-v3/LN-100003.pdf

📝 Filling Promissory Note for Loan LN-100004
✅ Uploaded flattened PDF as promissory-note-filled-v3/LN-100004.pdf

📝 Filling Promissory Note for Loan LN-100005
✅ Uploaded flattened PDF as promissory-note-filled-v3/LN-100005.pdf

📝 Filling Promissory Note for Loan LN-100006
✅ Uploaded flattened PDF as promissory-note-filled-v3/LN-100006.pdf

📝 Filling Promissory Note for Loan LN-100007
✅ Uploaded flattened PDF as promissory-note-filled-v3/LN-100007.pdf

📝 Filling Promissory Note for Loan LN-100008
✅ Uploaded flattened PDF as promissory-not

In [31]:
import os
from io import BytesIO
import pandas as pd
from pdfrw import PdfReader, PdfWriter, PageMerge
from reportlab.pdfgen import canvas
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv

# --- Load environment variables ---
load_dotenv("environment.env")
connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
sample_container_name = os.getenv("SAMPLE_CONTAINER_NAME")  # CSV container
template_container_name = "fillable-pdfs"                  # template PDFs
output_container_name = "promissory-note-filled-v3"        # output PDFs

blob_service_client = BlobServiceClient.from_connection_string(connection_string)

# --- Load CSV ---
csv_blob_name = "Promissory_Note_Dummy_Data__50_rows_.csv"
blob_client = blob_service_client.get_blob_client(container=sample_container_name, blob=csv_blob_name)
data = blob_client.download_blob().readall()
df = pd.read_csv(BytesIO(data))

# --- Helper: get all form fields with their page number and coordinates ---
def get_form_fields(template_pdf):
    fields = {}
    for page_index, page in enumerate(template_pdf.pages):
        if page.Annots:
            for annot in page.Annots:
                if annot.Subtype and annot.Subtype == "/Widget":
                    field_name = str(annot.T)[1:-1]  # remove parentheses
                    rect = [float(x) for x in annot.Rect]
                    fields[field_name] = (page_index, rect)
    return fields

# --- Helper: create overlay PDF with all fields filled ---
def create_overlay(fields_to_fill, template_pdf):
    form_fields = get_form_fields(template_pdf)
    overlays = []

    for page_index, page in enumerate(template_pdf.pages):
        packet = BytesIO()
        width = float(page.MediaBox[2])
        height = float(page.MediaBox[3])
        c = canvas.Canvas(packet, pagesize=(width, height))

        # Fill all fields present on this page
        for field_name, (f_page_index, rect) in form_fields.items():
            if f_page_index == page_index:
                value = fields_to_fill.get(field_name, "")  # default empty if missing
                x0, y0, x1, y1 = rect
                c.drawString(x0 + 2, y0 + 2, str(value))  # simple positioning

        c.showPage()
        c.save()
        packet.seek(0)
        overlays.append(packet)

    return overlays

# --- Process each row in CSV ---
for _, row in df.iterrows():
    loan_number = str(row.get("Loan Number", "unknown"))
    print(f"Filling PDF for Loan {loan_number}")

    # Download template PDF
    template_blob_client = blob_service_client.get_blob_client(container=template_container_name, blob="Promissory Note-FORM.pdf")
    template_pdf_data = template_blob_client.download_blob().readall()
    template_pdf = PdfReader(BytesIO(template_pdf_data))

    # Prepare dictionary of field values
    fields_to_fill = {col: str(row[col]) if pd.notna(row[col]) else "" for col in df.columns}

    # Create overlay PDFs per page
    overlay_streams = create_overlay(fields_to_fill, template_pdf)
    overlay_pdfs = [PdfReader(s) for s in overlay_streams]

    # Merge overlay with template
    for page_index, page in enumerate(template_pdf.pages):
        overlay_page = overlay_pdfs[page_index].pages[0]
        PageMerge(page).add(overlay_page).render()

    # Save flattened PDF to memory
    output_stream = BytesIO()
    PdfWriter().write(output_stream, template_pdf)
    output_stream.seek(0)

    # Upload to Azure Blob Storage
    output_blob_name = f"{loan_number}.pdf"
    output_blob_client = blob_service_client.get_blob_client(container=output_container_name, blob=output_blob_name)
    output_blob_client.upload_blob(output_stream, overwrite=True)
    print(f"✅ Uploaded {output_blob_name} to {output_container_name}")


Filling PDF for Loan LN-100000
✅ Uploaded LN-100000.pdf to promissory-note-filled-v3
Filling PDF for Loan LN-100001
✅ Uploaded LN-100001.pdf to promissory-note-filled-v3
Filling PDF for Loan LN-100002
✅ Uploaded LN-100002.pdf to promissory-note-filled-v3
Filling PDF for Loan LN-100003
✅ Uploaded LN-100003.pdf to promissory-note-filled-v3
Filling PDF for Loan LN-100004
✅ Uploaded LN-100004.pdf to promissory-note-filled-v3
Filling PDF for Loan LN-100005
✅ Uploaded LN-100005.pdf to promissory-note-filled-v3
Filling PDF for Loan LN-100006
✅ Uploaded LN-100006.pdf to promissory-note-filled-v3
Filling PDF for Loan LN-100007
✅ Uploaded LN-100007.pdf to promissory-note-filled-v3
Filling PDF for Loan LN-100008
✅ Uploaded LN-100008.pdf to promissory-note-filled-v3
Filling PDF for Loan LN-100009
✅ Uploaded LN-100009.pdf to promissory-note-filled-v3
Filling PDF for Loan LN-100010
✅ Uploaded LN-100010.pdf to promissory-note-filled-v3
Filling PDF for Loan LN-100011
✅ Uploaded LN-100011.pdf to promis

In [32]:
import os
from io import BytesIO
import pandas as pd
from pdfrw import PdfReader, PdfWriter, PageMerge, PdfName
from reportlab.pdfgen import canvas
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv

# --- Load environment variables ---
load_dotenv("environment.env")
connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
sample_container_name = os.getenv("SAMPLE_CONTAINER_NAME")
template_container_name = "fillable-pdfs"
output_container_name = "promissory-note-filled-v3"

blob_service_client = BlobServiceClient.from_connection_string(connection_string)

# --- Load CSV ---
csv_blob_name = "Promissory_Note_Dummy_Data__50_rows_.csv"
blob_client = blob_service_client.get_blob_client(container=sample_container_name, blob=csv_blob_name)
data = blob_client.download_blob().readall()
df = pd.read_csv(BytesIO(data))

# --- Helper: get all form fields with page index and coordinates ---
def get_form_fields(template_pdf):
    fields = {}
    for page_index, page in enumerate(template_pdf.pages):
        if page.Annots:
            for annot in page.Annots:
                if annot.Subtype and annot.Subtype == "/Widget":
                    field_name = str(annot.T)[1:-1] if annot.T else None
                    rect = [float(x) for x in annot.Rect] if annot.Rect else None
                    if field_name and rect:
                        fields[field_name] = (page_index, rect)
    return fields

# --- Helper: dynamically adjust font to fit inside rectangle ---
def draw_text_in_rect(c, text, rect, max_font=12, min_font=4):
    x0, y0, x1, y1 = rect
    field_width = x1 - x0 - 2
    field_height = y1 - y0 - 2
    font_size = max_font
    c.setFont("Helvetica", font_size)

    # Reduce font until text fits width
    while font_size > min_font and c.stringWidth(text, "Helvetica", font_size) > field_width:
        font_size -= 0.5
        c.setFont("Helvetica", font_size)

    # Draw text slightly above bottom-left corner
    c.drawString(x0 + 1, y0 + (field_height - font_size)/2, text)

# --- Helper: create overlay PDF with all fields filled ---
def create_overlay(fields_to_fill, template_pdf):
    form_fields = get_form_fields(template_pdf)
    overlays = []

    for page_index, page in enumerate(template_pdf.pages):
        packet = BytesIO()
        width = float(page.MediaBox[2])
        height = float(page.MediaBox[3])
        c = canvas.Canvas(packet, pagesize=(width, height))

        # Fill all fields present on this page
        for field_name, (f_page_index, rect) in form_fields.items():
            if f_page_index == page_index:
                value = str(fields_to_fill.get(field_name, ""))
                draw_text_in_rect(c, value, rect)

        c.showPage()
        c.save()
        packet.seek(0)
        overlays.append(packet)

    return overlays

# --- Process each row in CSV ---
for _, row in df.iterrows():
    loan_number = str(row.get("Loan Number", "unknown"))
    print(f"Filling PDF for Loan {loan_number}")

    # Download template PDF
    template_blob_client = blob_service_client.get_blob_client(
        container=template_container_name,
        blob="Promissory Note-FORM.pdf"
    )
    template_pdf_data = template_blob_client.download_blob().readall()
    template_pdf = PdfReader(BytesIO(template_pdf_data))

    # Map CSV columns to field values
    fields_to_fill = {col: str(row[col]) if pd.notna(row[col]) else "" for col in df.columns}

    # Create overlay PDFs per page with auto font
    overlay_streams = create_overlay(fields_to_fill, template_pdf)
    overlay_pdfs = [PdfReader(s) for s in overlay_streams]

    # Merge overlay with template
    for page_index, page in enumerate(template_pdf.pages):
        overlay_page = overlay_pdfs[page_index].pages[0]
        PageMerge(page).add(overlay_page).render()

    # Save flattened PDF to memory
    output_stream = BytesIO()
    PdfWriter().write(output_stream, template_pdf)
    output_stream.seek(0)

    # Upload to Azure Blob Storage
    output_blob_name = f"{loan_number}.pdf"
    output_blob_client = blob_service_client.get_blob_client(container=output_container_name, blob=output_blob_name)
    output_blob_client.upload_blob(output_stream, overwrite=True)
    print(f"✅ Uploaded {output_blob_name} to {output_container_name}")


Filling PDF for Loan LN-100000
✅ Uploaded LN-100000.pdf to promissory-note-filled-v3
Filling PDF for Loan LN-100001
✅ Uploaded LN-100001.pdf to promissory-note-filled-v3
Filling PDF for Loan LN-100002
✅ Uploaded LN-100002.pdf to promissory-note-filled-v3
Filling PDF for Loan LN-100003
✅ Uploaded LN-100003.pdf to promissory-note-filled-v3
Filling PDF for Loan LN-100004
✅ Uploaded LN-100004.pdf to promissory-note-filled-v3
Filling PDF for Loan LN-100005
✅ Uploaded LN-100005.pdf to promissory-note-filled-v3
Filling PDF for Loan LN-100006
✅ Uploaded LN-100006.pdf to promissory-note-filled-v3
Filling PDF for Loan LN-100007
✅ Uploaded LN-100007.pdf to promissory-note-filled-v3
Filling PDF for Loan LN-100008
✅ Uploaded LN-100008.pdf to promissory-note-filled-v3
Filling PDF for Loan LN-100009
✅ Uploaded LN-100009.pdf to promissory-note-filled-v3
Filling PDF for Loan LN-100010
✅ Uploaded LN-100010.pdf to promissory-note-filled-v3
Filling PDF for Loan LN-100011
✅ Uploaded LN-100011.pdf to promis